# Belief Evolution — Watch Your Posteriors Converge to Truth

**Goal:** Visualize how Thompson Sampling's posterior beliefs evolve from uncertainty to confidence as data accumulates.

**Time:** 15 minutes

**What you'll learn:**
- How posteriors concentrate around true values over time
- Why strong priors slow convergence
- How to track "probability of selecting each arm" to understand exploration decay

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

sns.set_style('whitegrid')
np.random.seed(42)

In [ ]:
learning_objectives(["**Posteriors concentrate over time** — Uncertainty (wide distributions) → Confidence (narrow distributions)", "**Exploration decays naturally** — Wide posteriors → diverse samples → exploration. Narrow posteriors → concentrated samples → exploitation.", "**Prior strength = effective sample size** — Beta(α, β) acts like α + β - 2 prior observations. Strong priors slow learning.", "**Selection probabilities track convergence** — As posteriors separate, the best arm gets selected more often (but never 100%)."])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Setup: A Difficult Bandit Problem

Three commodity sectors with very close true returns — hard to distinguish.

In [ ]:
# True win rates (very close together)
true_probs = np.array([0.45, 0.50, 0.55])
sector_names = ['Energy', 'Metals', 'Agriculture']

print("True win rates:")
for name, prob in zip(sector_names, true_probs):
    print(f"  {name:12s}: {prob:.2f}")

print("\nChallenge: Agriculture is best, but only by 5 percentage points.")
print("How long to confidently identify it?")

## Run Thompson Sampling and Track Posteriors

We'll save the posterior parameters at every round to visualize evolution.

In [ ]:
def run_thompson_with_history(true_probs, T=5000, alpha_0=1, beta_0=1):
    """Run Thompson Sampling and save posterior history."""
    K = len(true_probs)
    alpha = np.ones(K) * alpha_0
    beta_param = np.ones(K) * beta_0
    
    # History
    alpha_history = np.zeros((T+1, K))
    beta_history = np.zeros((T+1, K))
    alpha_history[0] = alpha.copy()
    beta_history[0] = beta_param.copy()
    
    selections = np.zeros(T, dtype=int)
    
    for t in range(T):
        # Sample and select
        samples = np.random.beta(alpha, beta_param)
        arm = np.argmax(samples)
        selections[t] = arm
        
        # Observe
        reward = np.random.binomial(1, true_probs[arm])
        
        # Update
        if reward == 1:
            alpha[arm] += 1
        else:
            beta_param[arm] += 1
        
        # Save
        alpha_history[t+1] = alpha.copy()
        beta_history[t+1] = beta_param.copy()
    
    return alpha_history, beta_history, selections

# Run with weak prior
alpha_hist, beta_hist, selections = run_thompson_with_history(true_probs, T=5000)
print("Simulation complete. Posteriors tracked over 5000 rounds.")

## Visualize Posterior Evolution at Key Milestones

In [ ]:
def plot_posteriors_at_round(alpha_hist, beta_hist, round_idx, true_probs, sector_names):
    """Plot Beta posteriors at specific round."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    x = np.linspace(0, 1, 300)
    
    for i, (ax, name) in enumerate(zip(axes, sector_names)):
        alpha_i = alpha_hist[round_idx, i]
        beta_i = beta_hist[round_idx, i]
        
        y = beta.pdf(x, alpha_i, beta_i)
        ax.fill_between(x, y, alpha=0.3, label=f'Beta({int(alpha_i)}, {int(beta_i)})')
        ax.plot(x, y, linewidth=2)
        ax.axvline(true_probs[i], color='red', linestyle='--', linewidth=2, label=f'True: {true_probs[i]}')
        
        # Posterior mean
        post_mean = alpha_i / (alpha_i + beta_i)
        ax.axvline(post_mean, color='blue', linestyle=':', linewidth=2, label=f'Mean: {post_mean:.3f}')
        
        ax.set_title(f'{name}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Win Rate', fontsize=11)
        ax.set_ylabel('Density', fontsize=11)
        ax.legend(fontsize=9)
        ax.set_xlim(0.3, 0.7)
    
    plt.suptitle(f'Posterior Beliefs at Round {round_idx}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Plot at key rounds
for round_t in [0, 100, 500, 1000, 5000]:
    plot_posteriors_at_round(alpha_hist, beta_hist, round_t, true_probs, sector_names)

**Observe the progression:**

- **Round 0:** All three sectors have identical uniform priors — total uncertainty
- **Round 100:** Posteriors starting to separate, but still overlapping significantly
- **Round 500:** Clear separation emerging, Agriculture (0.55) pulling ahead
- **Round 1000:** Posteriors much tighter, Agriculture clearly distinguished
- **Round 5000:** Very concentrated around true values, minimal overlap

**Key insight:** Even with a small gap (0.45 vs 0.55), posteriors eventually separate. But it takes hundreds of rounds — exploration is expensive in noisy environments.

## Track Selection Probabilities Over Time

Let's estimate "probability of selecting each arm" by tracking selection frequency in rolling windows.

In [ ]:
def compute_selection_probs(selections, window=100):
    """Compute rolling selection probabilities."""
    T = len(selections)
    K = int(selections.max() + 1)
    probs = np.zeros((T, K))
    
    for t in range(window, T):
        recent = selections[t-window:t]
        for k in range(K):
            probs[t, k] = (recent == k).mean()
    
    return probs

selection_probs = compute_selection_probs(selections, window=100)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

for i, name in enumerate(sector_names):
    ax.plot(selection_probs[100:, i], label=f'{name} (true={true_probs[i]:.2f})', linewidth=2)

ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Selection Probability (100-round window)', fontsize=12)
ax.set_title('How Often Each Sector is Selected Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("Final selection probabilities (last 500 rounds):")
for i, name in enumerate(sector_names):
    final_prob = (selections[-500:] == i).mean()
    print(f"  {name:12s}: {final_prob:.1%}")

**Interpretation:**
- Early rounds: All three sectors selected roughly equally (high exploration)
- Mid-game: Agriculture starts getting selected more often as evidence accumulates
- Late game: Agriculture dominates (~80-90% selection), but Energy and Metals still get occasional pulls

**Why not 100% Agriculture?** Thompson Sampling never fully stops exploring. There's always a small chance that Agriculture's sample is low and another sector's sample is high. This is a feature, not a bug — it helps adapt to non-stationarity.

## Experiment: How Do Priors Affect Convergence?

Use an interactive widget to control prior strength and see convergence speed.

In [ ]:
def compare_prior_strengths(true_probs, prior_strengths, T=2000):
    """Run Thompson Sampling with different prior strengths."""
    fig, axes = plt.subplots(len(prior_strengths), 1, figsize=(12, 4*len(prior_strengths)))
    
    for idx, strength in enumerate(prior_strengths):
        # Run Thompson Sampling
        alpha_0 = strength
        beta_0 = strength
        alpha_hist, beta_hist, sels = run_thompson_with_history(
            true_probs, T=T, alpha_0=alpha_0, beta_0=beta_0
        )
        
        # Compute posterior means over time
        post_means = alpha_hist / (alpha_hist + beta_hist)
        
        ax = axes[idx] if len(prior_strengths) > 1 else axes
        for i, name in enumerate(sector_names):
            ax.plot(post_means[:, i], label=f'{name} (true={true_probs[i]:.2f})', linewidth=2)
            ax.axhline(true_probs[i], color='gray', linestyle='--', alpha=0.3)
        
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Posterior Mean', fontsize=11)
        ax.set_title(f'Prior Strength = {strength} (Beta({strength}, {strength}) starting prior)', 
                     fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(alpha=0.3)
        ax.set_ylim(0.3, 0.7)
    
    plt.tight_layout()
    plt.show()

# Compare weak, moderate, and strong priors
compare_prior_strengths(true_probs, prior_strengths=[1, 10, 50], T=2000)

**Key observation:**

- **Beta(1,1):** Converges quickly — data dominates from the start
- **Beta(10,10):** Slower convergence — prior acts like 18 prior observations
- **Beta(50,50):** Very slow convergence — prior acts like 98 prior observations, overwhelms early data

**For bandits, weak priors (Beta(1,1)) usually win** because you want to learn fast. Strong priors only make sense if you have genuine prior knowledge (e.g., from fundamental analysis).

## Modify This: Interactive Prior Exploration

Try different prior strengths and observe how convergence changes.

In [ ]:
# EXPERIMENT: Change prior_strength and re-run
prior_strength = 5  # Try: 1, 5, 10, 20, 100

alpha_hist, beta_hist, sels = run_thompson_with_history(
    true_probs, T=2000, alpha_0=prior_strength, beta_0=prior_strength
)

# Plot posterior means
post_means = alpha_hist / (alpha_hist + beta_hist)

plt.figure(figsize=(12, 6))
for i, name in enumerate(sector_names):
    plt.plot(post_means[:, i], label=f'{name}', linewidth=2)
    plt.axhline(true_probs[i], color='gray', linestyle='--', alpha=0.3)

plt.xlabel('Round', fontsize=12)
plt.ylabel('Posterior Mean Win Rate', fontsize=12)
plt.title(f'Convergence with Prior Strength = {prior_strength}', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.ylim(0.3, 0.7)
plt.tight_layout()
plt.show()

# Question: At what round do posteriors get within 0.02 of true values?
converged = np.where(np.abs(post_means - true_probs).max(axis=1) < 0.02)[0]
if len(converged) > 0:
    print(f"Converged to within 0.02 by round {converged[0]}")
else:
    print("Did not converge within 0.02 in 2000 rounds — prior too strong!")

## Summary: Key Insights on Belief Evolution

**What you learned:**

1. **Posteriors concentrate over time** — Uncertainty (wide distributions) → Confidence (narrow distributions)
2. **Exploration decays naturally** — Wide posteriors → diverse samples → exploration. Narrow posteriors → concentrated samples → exploitation.
3. **Prior strength = effective sample size** — Beta(α, β) acts like α + β - 2 prior observations. Strong priors slow learning.
4. **Selection probabilities track convergence** — As posteriors separate, the best arm gets selected more often (but never 100%).

**Commodity trading implications:**

- **Use weak priors unless you have strong views** — Beta(1,1) lets market data drive decisions
- **Track posterior evolution to gauge confidence** — Wide posteriors = low confidence = more exploration needed
- **Don't expect 100% allocation to best sector** — Thompson Sampling maintains small exploration even late in the game (good for regime changes)
- **Convergence takes time with close alternatives** — If sector returns are within 5%, expect hundreds of rounds to confidently separate them

**Next steps:**
- `03_gaussian_thompson_commodities.ipynb` — Apply Thompson Sampling to real commodity return data (Gaussian rewards, not Bernoulli)

In [ ]:
key_takeaways(["Use weak priors unless you have strong views", "Track posterior evolution to gauge confidence", "Don't expect 100% allocation to best sector", "Convergence takes time with close alternatives"])